In [61]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

target_files = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}
docs_q1 = [d for d in documents if d["filename"] in target_files]
len(docs_q1)  

3

In [62]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv("../Lesson1/.env")
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print(os.getenv("BASE_URL"))

https://models.inference.ai.azure.com


In [63]:
import json
from pydantic import BaseModel
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

input_tokens = []

for doc in docs_q1:
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    input_tokens.append(usage.prompt_tokens)
    print(doc["filename"], "->", result.questions[:2])

sum(input_tokens) / len(input_tokens)

01-agentic-rag/lessons/01-intro.md -> ['What is the main focus of this lesson regarding RAG systems?', 'How do we implement LLMs in this course without getting into the theory?']
01-agentic-rag/lessons/02-environment.md -> ['What are the prerequisites needed for this module?', 'How do I create a new project for this course?']
01-agentic-rag/lessons/03-rag.md -> ['What does RAG stand for and what does it involve?', 'How can we improve the context provided to the LLM for better answers?']


1358.0

In [64]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [65]:
from gitsource import chunk_documents


chunks = chunk_documents(documents, size=2000, step=1000)


len(chunks)

295

In [66]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [80]:
def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results
    )

In [81]:
q = ground_truth[0]["question"]

text_results = text_index.search(q)

text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [82]:
from embedder import Embedder
import numpy as np
from tqdm.auto import tqdm
from minsearch import VectorSearch

In [83]:
model = Embedder()


In [84]:
vectors = []

for chunk in tqdm(chunks):
    vector = model.encode(chunk["content"])
    vectors.append(vector)

X = np.array(vectors)

  0%|          | 0/295 [00:00<?, ?it/s]

In [85]:
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [86]:
def vector_search(query, num_results=5):
    query_vector = model.encode(query)

    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [87]:
vector_results = vector_search(q)

vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [88]:
def compute_relevance(record, search_function):
    question = record["question"]
    correct_filename = record["filename"]

    results = search_function(question)

    relevance = []

    for result in results:
        is_relevant = result["filename"] == correct_filename
        relevance.append(int(is_relevant))

    return relevance

In [89]:
def hit_rate(relevance_total):
    cnt = 0

    for relevance in relevance_total:
        if 1 in relevance:
            cnt = cnt + 1

    return cnt / len(relevance_total)

In [90]:
def mrr(relevance_total):
    total_score = 0.0

    for relevance in relevance_total:
        for rank, rel in enumerate(relevance):
            if rel == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [91]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for record in tqdm(ground_truth):
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [92]:
text_metrics = evaluate(ground_truth, text_search)
text_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [93]:
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [94]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [95]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [96]:
for k in [1, 50, 100, 200]:
    def search_fn(query, k=k):
        return hybrid_search(query, k=k)

    metrics = evaluate(ground_truth, search_fn)
    print(k, metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
